# Deep learning segmentation of spectrogram in a supervised setting

# load Dataset

In [1]:
process = False

In [2]:
import os
import numpy as np
import json
from pathlib import Path

def load_mask_with_spectrograms(path: str | Path) -> dict:
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


if process == True:

    # List all elements (files and folders)
    elements = os.listdir('data_mask_spectro/')

    X_eeg =  []
    X_spec = []
    Y = []

    for elem in elements:
        if '.json' in elem:
            # load masks and spectrogram
            data = load_mask_with_spectrograms('data_mask_spectro/' + elem)
            # load recording (use try except since can be from 2 different folders)
            try:
                y = np.load('anesthesia_database/' + data['recording'])
            except:
                y = np.load('anesthesia_database_Trousseau/' + data['recording'])
            # load window key names
            list_windows_keys = list(data['windows'].keys())
            # iterate over all windows
            for i in range(len(data['windows'])):
                current_window = data['windows'][list_windows_keys[i]]
                start_s = float(current_window["window_start_s"])
                end_s = float(current_window["window_end_s"])
                fs = int(current_window["fs_hz"])

                start_i = int(round(start_s * fs))
                end_i = int(round(end_s * fs))

                signal = y[start_i : end_i]
                t_signal = np.arange(len(signal)) / fs

                mask = current_window['mask']

                t_spec = current_window['t_spec']
                f_spec = current_window['f_spec']
                spec = np.array(current_window['spectrogram'])

                X_eeg.append(signal)
                X_spec.append(spec)
                Y.append(mask)


    #--- convert to np arrays
    X_eeg = np.array(X_eeg)
    X_spec = np.array(X_spec)
    Y = np.array(Y)

    np.save('X_Y_dataset/X_eeg', X_eeg)
    np.save('X_Y_dataset/X_spec', X_spec)
    np.save('X_Y_dataset/Y', Y)

else:
    X_eeg =  np.load('X_Y_dataset/X_eeg.npy')
    X_spec =  np.load('X_Y_dataset/X_spec.npy')
    Y = np.load('X_Y_dataset/Y.npy')

# Utilities

In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time
from tqdm.auto import tqdm

# -------------------------------------------------
# 1) Train / Val / Test split (classic, no stratify)
# -------------------------------------------------
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    """
    val_size is the fraction of the *remaining* data after removing test.
    Example: test_size=0.2, val_size=0.2 -> train=64%, val=16%, test=20%.
    """
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=None
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_size,
        random_state=random_state,
        stratify=None
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# ----------------------------
# Dataset: numpy -> torch
# X: (N, 45, 297) float32
# y: (N, 297) int64 in [0..9]
# ----------------------------
class SpectrogramSegDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        assert X.ndim == 3 and X.shape[1] == 45 and X.shape[2] == 297
        assert y.ndim == 2 and y.shape[1] == 297
        self.X = X.astype(np.float32)
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])         # (45, 297)
        y = torch.from_numpy(self.y[idx])         # (297,)
        return x, y


# ----------------------------
# Dice metric (macro Dice)
# for multiclass segmentation over time axis
# pred: (B, T) int64
# target: (B, T) int64
# ----------------------------
@torch.no_grad()
def dice_score_multiclass(pred: torch.Tensor, target: torch.Tensor, num_classes: int, eps: float = 1e-6):
    # pred/target: (B, T)
    B, T = pred.shape
    pred_oh = F.one_hot(pred, num_classes=num_classes).float()       # (B, T, C)
    tgt_oh  = F.one_hot(target, num_classes=num_classes).float()     # (B, T, C)

    # Sum over batch and time
    pred_sum = pred_oh.sum(dim=(0, 1))     # (C,)
    tgt_sum  = tgt_oh.sum(dim=(0, 1))      # (C,)
    inter    = (pred_oh * tgt_oh).sum(dim=(0, 1))  # (C,)

    dice_c = (2.0 * inter + eps) / (pred_sum + tgt_sum + eps)  # (C,)
    macro = dice_c.mean().item()
    return macro, dice_c.cpu().numpy()


# ----------------------------
# Differentiable Dice loss
# logits: (B, T, C)
# target: (B, T)
# ----------------------------
def dice_loss_from_logits(logits: torch.Tensor, target: torch.Tensor, num_classes: int, eps: float = 1e-6):
    probs = torch.softmax(logits, dim=-1)  # (B, T, C)
    tgt_oh = F.one_hot(target, num_classes=num_classes).float()  # (B, T, C)

    # sum over B,T
    inter = (probs * tgt_oh).sum(dim=(0, 1))       # (C,)
    p_sum = probs.sum(dim=(0, 1))                  # (C,)
    t_sum = tgt_oh.sum(dim=(0, 1))                 # (C,)

    dice = (2.0 * inter + eps) / (p_sum + t_sum + eps)  # (C,)
    return 1.0 - dice.mean()  # scalar


class CEDiceLoss(nn.Module):
    def __init__(self, num_classes: int, dice_weight: float = 0.5, class_weights: torch.Tensor | None = None):
        super().__init__()
        self.num_classes = num_classes
        self.dice_weight = dice_weight
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, logits: torch.Tensor, target: torch.Tensor):
        # logits (B,T,C) -> CE wants (B,C,T)
        ce = self.ce(logits.permute(0, 2, 1), target)
        dl = dice_loss_from_logits(logits, target, self.num_classes)
        return (1.0 - self.dice_weight) * ce + self.dice_weight * dl


# ----------------------------
# Simple train/eval loops
# ----------------------------
def run_one_epoch(model, loader, optimizer, loss_fn, device, num_classes: int, train: bool, desc: str = ""):
    model.train(train)
    total_loss = 0.0
    all_macro_dice = []
    n_seen = 0

    if device.startswith("cuda"):
        torch.cuda.synchronize()
    start = time.perf_counter()

    pbar = tqdm(loader, desc=desc, leave=False)
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        bs = x.size(0)
        n_seen += bs

        with torch.set_grad_enabled(train):
            logits = model(x)  # (B, T, C)
            loss = loss_fn(logits, y)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * bs

        pred = logits.argmax(dim=-1)  # (B, T)
        macro_dice, _ = dice_score_multiclass(pred, y, num_classes=num_classes)
        all_macro_dice.append(macro_dice)

        # running stats for display
        run_loss = total_loss / n_seen
        run_dice = float(np.mean(all_macro_dice))
        pbar.set_postfix(loss=f"{run_loss:.4f}", dice=f"{run_dice:.4f}")

    if device.startswith("cuda"):
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    avg_loss = total_loss / len(loader.dataset)
    avg_macro_dice = float(np.mean(all_macro_dice)) if all_macro_dice else float("nan")
    return avg_loss, avg_macro_dice, elapsed



def compute_confusion_matrix(model, loader, device, num_classes):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = logits.argmax(dim=-1)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds).reshape(-1)
    all_targets = np.concatenate(all_targets).reshape(-1)

    cm = confusion_matrix(
        all_targets,
        all_preds,
        labels=list(range(num_classes))
    )

    return cm


def plot_confusion_matrix(cm, normalize=True):
    """
    cm: (C, C)
    """
    if normalize:
        cm = cm.astype(np.float64)
        row_sums = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, row_sums, where=row_sums != 0)

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f" if normalize else "d",
        cmap="Blues",
        xticklabels=[f"P{i}" for i in range(cm.shape[0])],
        yticklabels=[f"T{i}" for i in range(cm.shape[0])]
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix" + (" (Normalized)" if normalize else ""))
    plt.tight_layout()
    plt.show()



## Model {22D-CN encoder  +  1D temporal head}

_Temporal head is the decoder here ?_

In [6]:
class ConvBlock2D(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, padding=p),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=k, padding=p),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.net(x)


class TemporalHead1D(nn.Module):
    """
    Input: (B, F, T) features
    Output: (B, T, C)
    """
    def __init__(self, in_ch, hidden=128, num_classes=10, depth=4, dropout=0.1):
        super().__init__()
        layers = []
        ch = in_ch
        for i in range(depth):
            layers += [
                nn.Conv1d(ch, hidden, kernel_size=5, padding=2, dilation=1),
                nn.BatchNorm1d(hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            ]
            ch = hidden
        self.backbone = nn.Sequential(*layers)
        self.classifier = nn.Conv1d(hidden, num_classes, kernel_size=1)

    def forward(self, x):
        # x: (B, F, T)
        h = self.backbone(x)           # (B, hidden, T)
        out = self.classifier(h)       # (B, C, T)
        return out.permute(0, 2, 1)    # (B, T, C)


class BaselineSpecSeg(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Input x: (B, 45, 297)
        self.enc = nn.Sequential(
            ConvBlock2D(1, 32),
            nn.MaxPool2d(kernel_size=(2, 2)),   # freq 45->22, time 297->148
            ConvBlock2D(32, 64),
            nn.MaxPool2d(kernel_size=(2, 2)),   # freq 22->11, time 148->74
            ConvBlock2D(64, 128),
        )
        # We will reduce freq by mean pooling -> (B, 128, T')
        self.head = TemporalHead1D(in_ch=128, hidden=128, num_classes=num_classes, depth=4)

    def forward(self, x):
        # x: (B, 45, 297)
        x = x.unsqueeze(1)        # (B, 1, 45, 297)
        h = self.enc(x)           # (B, 128, F', T') with F'≈11, T'≈74
        h = h.mean(dim=2)         # (B, 128, T')
        # upsample to T=297
        h = F.interpolate(h, size=297, mode="linear", align_corners=False)  # (B, 128, 297)
        logits = self.head(h)     # (B, 297, C)
        return logits
    


# NOTE:  what  are logits
# NOTE:  should i  add  a layer in aselineSpecSEg


#  Model {CNN14 (PANNS) feature extractor (pretrained) + temporal head}

In [8]:
class CNN14_PANNs_FeatureSeg(nn.Module):
    """
    Uses PANNs CNN14 via panns_inference SoundEventDetection.
    We pull *framewise* features, then learn a temporal head to predict your 10 classes.

    IMPORTANT:
    - PANNs expects raw audio at 32kHz in the reference usage.
    - If you ONLY have spectrograms (45x297), pretrained PANNs won't match perfectly.
      Best is to keep raw audio and generate features inside the model.

    Below I provide two forward paths:
    (A) forward_from_audio: recommended (raw waveform)
    (B) forward_from_spectrogram: fallback (learnable adapter to mimic expected shape)
    """

    def __init__(self, num_classes=10, device="cuda"):
        super().__init__()
        self.num_classes = num_classes
        self.device_str = device

        # Lazy import so code doesn't crash if you only run baseline/AST
        import panns_inference
        from panns_inference import SoundEventDetection

        self.sed = SoundEventDetection(checkpoint_path=None, device=device)
        # self.sed.inference(audio) returns framewise_output for AudioSet classes (527)
        # We'll treat that as framewise features, then project to 10 classes.

        # A small temporal projection head: 527 -> hidden -> 10
        self.proj = nn.Sequential(
            nn.Conv1d(527, 256, kernel_size=1),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Conv1d(256, num_classes, kernel_size=1),
        )

    @torch.no_grad()
    def _panns_framewise_from_audio(self, audio_32k: np.ndarray) -> torch.Tensor:
        """
        audio_32k: numpy float waveform (B, samples) sampled at 32000Hz
        returns: torch float tensor (B, T_panns, 527)
        """
        framewise = self.sed.inference(audio_32k)  # numpy (B, T_panns, 527)
        return torch.from_numpy(framewise).float()

    def forward_from_audio(self, audio_32k: np.ndarray) -> torch.Tensor:
        """
        Returns logits (B, 297, 10)
        """
        feats = self._panns_framewise_from_audio(audio_32k).to(next(self.parameters()).device)  # (B, Tp, 527)
        feats = feats.permute(0, 2, 1)  # (B, 527, Tp)

        # project to 10
        logits = self.proj(feats)  # (B, 10, Tp)

        # upsample to 297 frames to match your mask length
        logits = F.interpolate(logits, size=297, mode="linear", align_corners=False)  # (B, 10, 297)
        return logits.permute(0, 2, 1)  # (B, 297, 10)

    def forward_from_spectrogram(self, x_spec: torch.Tensor) -> torch.Tensor:
        """
        Fallback if you *only* have spectrograms.
        Here we DO NOT use PANNs properly (since PANNs is pretrained on audio->logmel inside).
        So instead we learn a small adapter and still keep the same temporal head.

        x_spec: (B, 45, 297)
        returns logits: (B, 297, 10)
        """
        # Adapter: (45,297)-> "fake" 527-dim per frame features
        # (This is no longer truly pretrained CNN14; it's a compatibility fallback.)
        h = x_spec  # (B,45,297)
        h = nn.functional.layer_norm(h, normalized_shape=h.shape[1:])  # quick normalize
        h = h.permute(0, 2, 1)  # (B,297,45)

        # simple MLP to 527 dims to reuse same proj head
        h = h.reshape(-1, 45)
        h = F.gelu(nn.Linear(45, 256, device=h.device)(h))
        h = nn.Linear(256, 527, device=h.device)(h)
        h = h.view(x_spec.size(0), 297, 527).permute(0, 2, 1)  # (B,527,297)

        logits = self.proj(h)  # (B,10,297)
        return logits.permute(0, 2, 1)

    def forward(self, x):
        # For consistency with the baseline trainer, assume x is spectrogram.
        return self.forward_from_spectrogram(x)



# NOTE: where are the weights blocked for training on the CNN14  
# NOTE: doc says could enter as raw eeg too

# MODEL {AST  feature extractor (pretrained)  + temporal head}

In [9]:
from transformers import ASTModel, ASTConfig  # pip install transformers

class ASTFeatureSeg(nn.Module):
    """
    AST as a feature extractor + temporal head for 1D time segmentation.

    This is a *practical* adapter approach:
    - We resize spectrogram to (128, 1024) (mel x frames).
    - Feed to ASTModel (as if it were log-mel).
    - Convert token sequence -> time sequence -> upsample to 297 -> classify.

    You may want to try these checkpoints:
      - "MIT/ast-finetuned-audioset-10-10-0.4593" (common)
    (Exact checkpoint choice depends on availability; any ASTModel-compatible one works.)
    """
    def __init__(self, checkpoint="MIT/ast-finetuned-audioset-10-10-0.4593", num_classes=10):
        super().__init__()
        self.ast = ASTModel.from_pretrained(checkpoint)
        hidden = self.ast.config.hidden_size

        # temporal head on extracted time features
        self.temporal = nn.Sequential(
            nn.Conv1d(hidden, 256, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Conv1d(256, num_classes, kernel_size=1),
        )

    def forward(self, x):
        """
        x: (B, 45, 297)
        returns: logits (B, 297, 10)
        """
        B, Freq, T = x.shape

        # Resize to (B, 128, 1024) then add channel dim to match AST expectations
        # AST expects input_values shape (B, 1024, 128) in HF implementation for log-mel features.
        # We'll create (B, 128, 1024) then transpose.
        x2 = x.unsqueeze(1)  # (B,1,45,297)
        x2 = F.interpolate(x2, size=(128, 1024), mode="bilinear", align_corners=False)  # (B,1,128,1024)
        x2 = x2.squeeze(1)  # (B,128,1024)
        input_values = x2.transpose(1, 2)  # (B,1024,128)

        out = self.ast(input_values=input_values)
        # out.last_hidden_state: (B, tokens, hidden)
        hs = out.last_hidden_state

        # Drop CLS token if present (AST uses a CLS token)
        # Many ViT-style models: first token is CLS
        hs = hs[:, 1:, :]  # (B, tokens-1, hidden)

        # We need a "time axis". Tokens correspond to 2D patches (time x freq).
        # We'll approximate:
        # - reshape tokens into (B, H_patches, W_patches, hidden) if possible,
        # - average over freq patches to get (B, W_patches, hidden) as time sequence.
        #
        # We infer patch grid sizes from config:
        patch = self.ast.config.patch_size  # usually 16
        # After resize: frames=1024, mel=128
        # patches along time ~ 1024/patch, along freq ~ 128/patch
        w = 1024 // patch
        h = 128 // patch
        tokens = hs.shape[1]
        # If mismatch due to stride/overlap specifics, fall back to treating tokens as 1D sequence
        if tokens == h * w:
            hs2 = hs.view(B, h, w, -1)          # (B,h,w,hidden)
            time_feat = hs2.mean(dim=1)         # (B,w,hidden)
        else:
            # fallback: just treat token sequence as time-like and interpolate later
            time_feat = hs                      # (B,tokens,hidden)

        # Now time_feat: (B, W', hidden)
        time_feat = time_feat.permute(0, 2, 1)  # (B, hidden, W')

        # Upsample to 297
        time_feat = F.interpolate(time_feat, size=297, mode="linear", align_corners=False)  # (B,hidden,297)

        logits = self.temporal(time_feat)  # (B,10,297)
        return logits.permute(0, 2, 1)     # (B,297,10)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

# Training and testing

In [ ]:
# -------------------------------------------------
# 2) Main training script
# -------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = 10

# X: (N,45,297), y: (N,297) numpy arrays
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X_spec, Y,
    test_size=0.2,
    val_size=0.2,
    random_state=42
)

print("Shapes:")
print("  Train:", X_train.shape, y_train.shape)
print("  Val:  ", X_val.shape, y_val.shape)
print("  Test: ", X_test.shape, y_test.shape)

# Build datasets/loaders
train_ds = SpectrogramSegDataset(X_train, y_train)
val_ds   = SpectrogramSegDataset(X_val, y_val)
test_ds  = SpectrogramSegDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# Pick one model:
model = BaselineSpecSeg(num_classes=num_classes).to(device)
# model = CNN14_PANNs_FeatureSeg(num_classes=num_classes, device=device).to(device)
# model = ASTFeatureSeg(num_classes=num_classes).to(device)

# -------------------------------------------------
# 3) Class weights from TRAIN ONLY (robust to missing classes)
# -------------------------------------------------
counts = np.bincount(y_train.reshape(-1), minlength=num_classes).astype(np.float64)
present = counts > 0

weights = np.zeros(num_classes, dtype=np.float64)
if present.any():
    inv = counts[present].sum() / (counts[present] + 1e-6)
    inv = inv / inv.mean()
    weights[present] = inv
# missing classes keep weight 0.0

print("Train class counts:", counts)
print("Train class weights:", weights)

class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

loss_fn = CEDiceLoss(num_classes=num_classes, dice_weight=0.5, class_weights=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

# -------------------------------------------------
# 4) Train using VAL for model selection (early stopping optional)
# -------------------------------------------------
best_val = -1.0
best_state = None

for epoch in range(1, 21):
    tr_loss, tr_dice = run_one_epoch(
        model, train_loader, optimizer, loss_fn, device, num_classes, train=True
    )
    va_loss, va_dice = run_one_epoch(
        model, val_loader, optimizer, loss_fn, device, num_classes, train=False
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {tr_loss:.4f} dice {tr_dice:.4f} | "
        f"val loss {va_loss:.4f} dice {va_dice:.4f}"
    )

    # Save best checkpoint by validation Dice
    if va_dice > best_val:
        best_val = va_dice
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# Load best model (by val dice)
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

# -------------------------------------------------
# 5) Final evaluation ONCE on TEST
# -------------------------------------------------
te_loss, te_dice = run_one_epoch(
    model, test_loader, optimizer, loss_fn, device, num_classes, train=False
)
print(f"\nFINAL TEST | loss {te_loss:.4f} dice {te_dice:.4f}")

# Plot confusion matrix
cm = compute_confusion_matrix(model, test_loader, device, num_classes)
plot_confusion_matrix(cm, normalize=True)


Shapes:
  Train: (265, 45, 297) (265, 297)
  Val:   (67, 45, 297) (67, 297)
  Test:  (84, 45, 297) (84, 297)
Train class counts: [43620.  3351.  7978.  3100.  8304.  5185.  1371.   165.  2606.  3025.]
Train class weights: [0.02669806 0.34752887 0.14597258 0.3756675  0.14024196 0.22460352
 0.84943053 7.05799542 0.44687999 0.38498157]
